# O3D - Physical Consistency Diagnostics

This notebook evaluates physical consistency of the O3A annual skill-weighted CMIP6 ensemble.

It is a defensible physics-informed diagnostic stage, not a full PINN training workflow.

Diagnostics:
- precipitation non-negativity,
- Tmax >= Tmin consistency,
- diurnal temperature range (DTR = Tmax - Tmin),
- scenario consistency of far-future warming,
- precipitation-temperature scaling by hydroclimatic zone,
- physical consistency score by scenario and zone.

Outputs are saved to `output/o3d_physical_consistency/`.

In [ ]:
# Cell 1 - Imports and configuration
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'output').exists() and (PROJECT_ROOT.parent / 'output').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

O3A_ROOT = PROJECT_ROOT / 'output' / 'o3a_skill_weighted_ensemble'
O3A_TABLE_DIR = O3A_ROOT / 'tables'
OUT_ROOT = PROJECT_ROOT / 'output' / 'o3d_physical_consistency'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
LOG_DIR = OUT_ROOT / 'logs'
for d in [TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VARIABLES = ['pr', 'tasmax', 'tasmin']
SCENARIOS = ['ssp245', 'ssp585']
HISTORICAL = 'historical'
FUTURE_PERIODS = ['near_future', 'mid_future', 'far_future']
BASELINE = 'baseline_1985_2014'

print(f'[INFO] Project root: {PROJECT_ROOT}')
print(f'[INFO] Output root: {OUT_ROOT}')

In [ ]:
# Cell 2 - Load O3A zone-level outputs
zone_annual = pd.read_csv(O3A_TABLE_DIR / 'o3a_zone_annual_ensemble_timeseries.csv')
period_changes = pd.read_csv(O3A_TABLE_DIR / 'o3a_period_changes_by_zone.csv')
weights = pd.read_csv(O3A_TABLE_DIR / 'o3a_model_pools_and_weights.csv')

# Keep ensemble mean for annual diagnostics.
annual_mean = zone_annual[zone_annual['stat'] == 'mean'].copy()
print(annual_mean.head())
print(period_changes.head())
print('[OK] O3A tables loaded')

In [ ]:
# Cell 3 - Annual physical consistency checks
rows = []

# Precipitation non-negativity by scenario/year/zone
pr = annual_mean[annual_mean['variable'] == 'pr'].copy()
for (scenario, zone), sub in pr.groupby(['scenario', 'zone']):
    vals = sub['value'].to_numpy(dtype=float)
    rows.append({
        'diagnostic': 'precipitation_non_negative',
        'scenario': scenario,
        'zone': zone,
        'n_years': len(vals),
        'n_violations': int(np.sum(vals < 0)),
        'violation_rate': float(np.mean(vals < 0)) if len(vals) else np.nan,
        'minimum_value': float(np.nanmin(vals)) if len(vals) else np.nan,
    })

# Tmax >= Tmin and DTR annual behavior
wide = annual_mean[annual_mean['variable'].isin(['tasmax', 'tasmin'])].pivot_table(
    index=['scenario', 'zone', 'year'], columns='variable', values='value'
).reset_index()
wide['dtr'] = wide['tasmax'] - wide['tasmin']
wide['tmax_lt_tmin'] = wide['tasmax'] < wide['tasmin']
wide.to_csv(TABLE_DIR / 'o3d_annual_temperature_consistency_timeseries.csv', index=False)

for (scenario, zone), sub in wide.groupby(['scenario', 'zone']):
    rows.append({
        'diagnostic': 'tmax_greater_equal_tmin',
        'scenario': scenario,
        'zone': zone,
        'n_years': len(sub),
        'n_violations': int(sub['tmax_lt_tmin'].sum()),
        'violation_rate': float(sub['tmax_lt_tmin'].mean()),
        'minimum_value': float(sub['dtr'].min()),
    })

annual_checks = pd.DataFrame(rows)
annual_checks.to_csv(TABLE_DIR / 'o3d_annual_physical_consistency_checks.csv', index=False)
print(annual_checks.head(20))
print(f'[OK] saved {TABLE_DIR / "o3d_annual_physical_consistency_checks.csv"}')

In [ ]:
# Cell 4 - Period DTR and scenario consistency diagnostics
# DTR period changes are derived from tasmax and tasmin changes.
tx = period_changes[period_changes['variable'] == 'tasmax'][['scenario','period','zone','absolute_change']].rename(columns={'absolute_change':'tasmax_change'})
tn = period_changes[period_changes['variable'] == 'tasmin'][['scenario','period','zone','absolute_change']].rename(columns={'absolute_change':'tasmin_change'})
dtr = tx.merge(tn, on=['scenario','period','zone'], how='inner')
dtr['dtr_change'] = dtr['tasmax_change'] - dtr['tasmin_change']
dtr.to_csv(TABLE_DIR / 'o3d_dtr_period_changes_by_zone.csv', index=False)

# Scenario consistency: SSP585 far-future warming should generally exceed SSP245 far-future warming.
scenario_rows = []
for var in ['tasmax', 'tasmin', 'pr']:
    sub = period_changes[(period_changes['variable'] == var) & (period_changes['period'] == 'far_future')]
    metric_col = 'percent_change_pr_only' if var == 'pr' else 'absolute_change'
    p245 = sub[sub['scenario'] == 'ssp245'][['zone', metric_col]].rename(columns={metric_col: 'ssp245_change'})
    p585 = sub[sub['scenario'] == 'ssp585'][['zone', metric_col]].rename(columns={metric_col: 'ssp585_change'})
    merged = p245.merge(p585, on='zone', how='inner')
    merged['scenario_consistent'] = merged['ssp585_change'] >= merged['ssp245_change']
    merged['variable'] = var
    scenario_rows.append(merged)
scenario_consistency = pd.concat(scenario_rows, ignore_index=True)
scenario_consistency.to_csv(TABLE_DIR / 'o3d_far_future_scenario_consistency.csv', index=False)

print(dtr.head())
print(scenario_consistency)
print('[OK] saved DTR and scenario consistency diagnostics')

In [ ]:
# Cell 5 - Precipitation-temperature scaling by zone and scenario
# Uses annual projected changes: relation between annual precipitation and mean temperature.
# Scaling is summarized as percent precipitation change per degC warming from baseline period means.
base = annual_mean[annual_mean['scenario'] == HISTORICAL]
base_p = base[base['variable'] == 'pr'].groupby('zone', as_index=False)['value'].mean().rename(columns={'value':'baseline_pr'})
base_tx = base[base['variable'] == 'tasmax'].groupby('zone', as_index=False)['value'].mean().rename(columns={'value':'baseline_tasmax'})
base_tn = base[base['variable'] == 'tasmin'].groupby('zone', as_index=False)['value'].mean().rename(columns={'value':'baseline_tasmin'})
base_t = base_tx.merge(base_tn, on='zone')
base_t['baseline_tmean'] = (base_t['baseline_tasmax'] + base_t['baseline_tasmin']) / 2
base_ref = base_p.merge(base_t[['zone','baseline_tmean']], on='zone')

rows = []
future = annual_mean[annual_mean['scenario'].isin(SCENARIOS)]
for (scenario, zone, year), grp in future.groupby(['scenario','zone','year']):
    vals = grp.set_index('variable')['value'].to_dict()
    if all(v in vals for v in ['pr','tasmax','tasmin']):
        b = base_ref[base_ref['zone'] == zone].iloc[0]
        tmean = (vals['tasmax'] + vals['tasmin']) / 2
        delta_t = tmean - b['baseline_tmean']
        pct_pr = ((vals['pr'] - b['baseline_pr']) / b['baseline_pr']) * 100 if b['baseline_pr'] != 0 else np.nan
        rows.append({'scenario':scenario,'zone':zone,'year':year,'delta_tmean':delta_t,'percent_pr_change':pct_pr})
scaling_annual = pd.DataFrame(rows)
scaling_annual.to_csv(TABLE_DIR / 'o3d_annual_precip_temperature_scaling_samples.csv', index=False)

summary_rows = []
for (scenario, zone), sub in scaling_annual.groupby(['scenario','zone']):
    if len(sub) >= 5 and sub['delta_tmean'].std() > 0:
        slope = np.polyfit(sub['delta_tmean'], sub['percent_pr_change'], 1)[0]
        r = np.corrcoef(sub['delta_tmean'], sub['percent_pr_change'])[0,1]
    else:
        slope = np.nan
        r = np.nan
    summary_rows.append({
        'scenario': scenario,
        'zone': zone,
        'n_years': len(sub),
        'percent_pr_per_degC': slope,
        'r_deltaT_pr': r,
        'mean_delta_tmean': sub['delta_tmean'].mean(),
        'mean_percent_pr_change': sub['percent_pr_change'].mean(),
    })
scaling_summary = pd.DataFrame(summary_rows)
scaling_summary.to_csv(TABLE_DIR / 'o3d_precip_temperature_scaling_by_zone.csv', index=False)
print(scaling_summary)
print('[OK] saved precipitation-temperature scaling diagnostics')

In [ ]:
# Cell 6 - Physical consistency score
# Score components by scenario/zone:
# 1 no precipitation negative values
# 2 no Tmax<Tmin violations
# 3 far-future SSP585 >= SSP245 for pr/tasmax/tasmin, assigned to zones
# 4 DTR stays positive annually
score_rows = []
annual_checks = pd.read_csv(TABLE_DIR / 'o3d_annual_physical_consistency_checks.csv')
scenario_consistency = pd.read_csv(TABLE_DIR / 'o3d_far_future_scenario_consistency.csv')
wide = pd.read_csv(TABLE_DIR / 'o3d_annual_temperature_consistency_timeseries.csv')

for scenario in [HISTORICAL] + SCENARIOS:
    zones = sorted(annual_mean['zone'].unique())
    for zone in zones:
        components = []
        pr_check = annual_checks[(annual_checks['diagnostic'] == 'precipitation_non_negative') & (annual_checks['scenario'] == scenario) & (annual_checks['zone'] == zone)]
        if len(pr_check):
            components.append(1 - float(pr_check['violation_rate'].iloc[0]))
        t_check = annual_checks[(annual_checks['diagnostic'] == 'tmax_greater_equal_tmin') & (annual_checks['scenario'] == scenario) & (annual_checks['zone'] == zone)]
        if len(t_check):
            components.append(1 - float(t_check['violation_rate'].iloc[0]))
        dtr_sub = wide[(wide['scenario'] == scenario) & (wide['zone'] == zone)]
        if len(dtr_sub):
            components.append(float((dtr_sub['dtr'] > 0).mean()))
        if scenario in SCENARIOS:
            sc = scenario_consistency[scenario_consistency['zone'] == zone]
            if len(sc):
                components.append(float(sc['scenario_consistent'].mean()))
        score_rows.append({
            'scenario': scenario,
            'zone': zone,
            'n_components': len(components),
            'physical_consistency_score': float(np.mean(components)) if components else np.nan,
        })
score = pd.DataFrame(score_rows)
score.to_csv(TABLE_DIR / 'o3d_physical_consistency_score_by_zone.csv', index=False)
print(score)
print('[OK] saved physical consistency score')

In [ ]:
# Cell 7 - Combined publication figures
annual_checks = pd.read_csv(TABLE_DIR / 'o3d_annual_physical_consistency_checks.csv')
dtr = pd.read_csv(TABLE_DIR / 'o3d_dtr_period_changes_by_zone.csv')
scenario_consistency = pd.read_csv(TABLE_DIR / 'o3d_far_future_scenario_consistency.csv')
scaling_summary = pd.read_csv(TABLE_DIR / 'o3d_precip_temperature_scaling_by_zone.csv')
score = pd.read_csv(TABLE_DIR / 'o3d_physical_consistency_score_by_zone.csv')

# Figure 1: physical consistency score, DTR change, scaling
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
score_pivot = score.pivot_table(index='zone', columns='scenario', values='physical_consistency_score')
score_pivot.plot(kind='bar', ax=axes[0], edgecolor='black', linewidth=0.4)
axes[0].set_title('A. Consistency score', loc='left', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_xlabel('Zone', fontweight='bold')
axes[0].grid(axis='y', linestyle=':', alpha=0.5)

dtr_far = dtr[dtr['period'] == 'far_future'].pivot_table(index='zone', columns='scenario', values='dtr_change')
dtr_far.plot(kind='bar', ax=axes[1], edgecolor='black', linewidth=0.4)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('B. Far-future DTR change', loc='left', fontweight='bold')
axes[1].set_ylabel('Change (degC)', fontweight='bold')
axes[1].set_xlabel('Zone', fontweight='bold')
axes[1].grid(axis='y', linestyle=':', alpha=0.5)

scale_pivot = scaling_summary.pivot_table(index='zone', columns='scenario', values='percent_pr_per_degC')
scale_pivot.plot(kind='bar', ax=axes[2], edgecolor='black', linewidth=0.4)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('C. Precipitation-temperature scaling', loc='left', fontweight='bold')
axes[2].set_ylabel('Precipitation change (% per degC)', fontweight='bold')
axes[2].set_xlabel('Zone', fontweight='bold')
axes[2].grid(axis='y', linestyle=':', alpha=0.5)
for ax in axes:
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
fig.savefig(FIG_DIR / 'physical_consistency_diagnostics_combined.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'physical_consistency_diagnostics_combined.pdf', bbox_inches='tight')
plt.show()

# Figure 2: scenario consistency heatmap
heat = scenario_consistency.pivot_table(index='zone', columns='variable', values='scenario_consistent', aggfunc='mean')
fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
im = ax.imshow(heat.values, aspect='auto', cmap='Greens', vmin=0, vmax=1)
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, fontweight='bold')
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontweight='bold')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, 'Yes' if heat.values[i,j] == 1 else 'No', ha='center', va='center', fontweight='bold')
ax.set_title('Far-future scenario ordering', fontweight='bold')
fig.colorbar(im, ax=ax, label='SSP5-8.5 >= SSP2-4.5')
fig.savefig(FIG_DIR / 'scenario_ordering_heatmap.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'scenario_ordering_heatmap.pdf', bbox_inches='tight')
plt.show()

print('[OK] figures saved')

In [ ]:
# Cell 8 - Completion summary
required = [
    TABLE_DIR / 'o3d_annual_physical_consistency_checks.csv',
    TABLE_DIR / 'o3d_annual_temperature_consistency_timeseries.csv',
    TABLE_DIR / 'o3d_dtr_period_changes_by_zone.csv',
    TABLE_DIR / 'o3d_far_future_scenario_consistency.csv',
    TABLE_DIR / 'o3d_precip_temperature_scaling_by_zone.csv',
    TABLE_DIR / 'o3d_physical_consistency_score_by_zone.csv',
    FIG_DIR / 'physical_consistency_diagnostics_combined.png',
    FIG_DIR / 'scenario_ordering_heatmap.png',
]
check = pd.DataFrame([{'path': str(p), 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0} for p in required])
check.to_csv(TABLE_DIR / 'o3d_completion_checklist.csv', index=False)
score = pd.read_csv(TABLE_DIR / 'o3d_physical_consistency_score_by_zone.csv')
score_summary = score.groupby('scenario', as_index=False)['physical_consistency_score'].mean()
score_table_lines = ['| scenario | mean_physical_consistency_score |', '|---|---:|']
for _, row in score_summary.iterrows():
    score_table_lines.append(f"| {row['scenario']} | {row['physical_consistency_score']:.3f} |")
score_table = '\n'.join(score_table_lines)
summary_lines = [
    '# O3D Physical Consistency Diagnostics Summary',
    '',
    '## Purpose',
    '',
    'This workflow evaluates physical consistency of the O3A annual skill-weighted ensemble. It is a physics-informed diagnostic step, not full PINN training.',
    '',
    '## Diagnostics',
    '',
    '- precipitation non-negativity',
    '- Tmax >= Tmin consistency',
    '- DTR behavior',
    '- SSP5-8.5 vs SSP2-4.5 far-future scenario ordering',
    '- precipitation-temperature scaling by zone',
    '- physical consistency score by zone',
    '',
    '## Mean Consistency Scores',
    '',
    score_table,
    '',
    '## Main Outputs',
    '',
    '- `tables/o3d_annual_physical_consistency_checks.csv`',
    '- `tables/o3d_dtr_period_changes_by_zone.csv`',
    '- `tables/o3d_far_future_scenario_consistency.csv`',
    '- `tables/o3d_precip_temperature_scaling_by_zone.csv`',
    '- `tables/o3d_physical_consistency_score_by_zone.csv`',
    '- `figures/physical_consistency_diagnostics_combined.png`',
    '- `figures/scenario_ordering_heatmap.png`',
    '',
    '## PINN Decision',
    '',
    'A full PINN was not trained in this stage because the current completed projection products are annual climate fields and do not include the full surface-energy-balance forcing set required for a defensible PINN. This stage provides physical consistency diagnostics that can support Objective 7; full PINN training should only be attempted after assembling appropriate radiative, turbulent flux, humidity, wind, and land-surface inputs.',
    '',
]
(OUT_ROOT / 'O3D_PHYSICAL_CONSISTENCY_SUMMARY.md').write_text('\n'.join(summary_lines), encoding='utf-8')
print(check)
print(f'[OK] saved {TABLE_DIR / "o3d_completion_checklist.csv"}')
print(f'[OK] saved {OUT_ROOT / "O3D_PHYSICAL_CONSISTENCY_SUMMARY.md"}')